# 00 · Data preparation & evaluation pipeline (P1)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syndr0/compute-efficient-genqa/blob/main/notebooks/00_data_and_evaluation.ipynb)

Builds every dataset used by the project from the MRQA release on the Hugging Face Hub:

* **training pools** – SQuAD, NewsQA, TriviaQA-web (MRQA *train* split)
* **dev sets** – held-out *passages* from each training pool (validation loss only)
* **ID test sets** – MRQA *validation* split of the three training sources
* **OOD test sets** – BioASQ and DuoRC-ParaphraseRC (MRQA *test* split)

then validates them, checks train/eval overlap, and unit-tests the EM/F1 metric.
Runtime: ~5 min (CPU is enough for this notebook).

In [ ]:
#@title Setup: clone the repo, install, choose where outputs go
#@markdown Outputs (data, adapters, predictions, metrics) persist on Google Drive so a
#@markdown disconnected runtime can resume: every stage skips work that is already saved.
USE_DRIVE = True  #@param {type:"boolean"}
SMOKE_TEST = False  #@param {type:"boolean"}
import os, sys
REPO_DIR = "/content/compute-efficient-genqa"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Syndr0/compute-efficient-genqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cegqa_outputs"
else:
    OUT = "/content/cegqa_outputs"
if SMOKE_TEST:
    OUT += "_smoke"
os.environ["CEGQA_OUTPUT_DIR"] = OUT
CONFIG = "--config configs/default.yaml" + (" --config configs/smoke.yaml" if SMOKE_TEST else "")
print("outputs ->", OUT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m cegqa.cli prepare $CONFIG

## Data report: sizes, lengths, answer-in-passage rate

In [ ]:
from cegqa.utils import load_config
cfgs = ["configs/default.yaml"] + (["configs/smoke.yaml"] if SMOKE_TEST else [])
cfg = load_config(cfgs)

In [ ]:
import json, pandas as pd
rep = json.load(open(f"{cfg['data_dir']}/data_report.json"))
rows = []
for split in ("train", "dev", "test"):
    for src, v in rep[split].items():
        rows.append({"split": split, "source": src, "n": v["n"],
                     "ctx_words_mean": round(v["context_words"]["mean"]), "ctx_words_p90": v["context_words"]["p90"],
                     "answer_words_mean": round(v["answer_words"]["mean"], 2),
                     "answer_in_context": f"{100*v['answer_in_context_rate']:.1f}%"})
pd.DataFrame(rows)

## Overlap checks (exact normalised match against the training pools)

In [ ]:
display(pd.DataFrame(rep["overlap_before_filter"]).T.rename_axis("before filtering"))
display(pd.DataFrame(rep["overlap_after_filter"]).T.rename_axis("after filtering"))

## Inspect examples and the prompt format

In [ ]:
from cegqa.data import load_split, load_fewshot
from cegqa.modeling import load_tokenizer
from cegqa.prompts import build_prompt
tok = load_tokenizer(cfg["model"]["name"])
ex = load_split(cfg, "test", "bioasq", limit=1)[0]
print("Q:", ex["question"], "| gold:", ex["answers"])
prompt, truncated = build_prompt(tok, ex, cfg["model"]["max_context_tokens"])
print(prompt[:1500])
print("few-shot demos:", [d["question"] for d in load_fewshot(cfg)])

## Metric unit tests (official SQuAD/MRQA normalisation)

In [ ]:
!pip install -q pytest && python -m pytest -q tests

In [ ]:
from cegqa.metrics import score
for pred, golds in [("the Eiffel Tower", ["Eiffel Tower"]), ("Paris, France", ["Paris"]), ("1945", ["in 1945"])]:
    print(f"{pred!r:22} vs {golds} -> {score(pred, golds)}")